In [8]:
import pandas as pd
from sqlalchemy import create_engine
import sys
import json

In [2]:
USER = "user"
PASSWORD = "userpassword"
HOST = "localhost"
PORT = "3306"
DB_NAME = "engineering_db"

In [3]:
engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}")

In [4]:
# список аналітичних запитів
queries = {
    "1. CAMPAIGN PERFORMANCE (Top 5 by CTR)": """
        SELECT c.campaign_name, top_c.CTR, top_c.total_cost from
        (SELECT e.campaign_id ,
        SUM(e.ad_cost) as total_cost,
        ROUND((SUM(case when ad_revenue > 0 then 1 else 0 END ) / NULLIF(COUNT(e.event_id), 0) * 100), 2) as CTR
        from events e
        WHERE e.event_timestamp >= '2024-11-01' AND e.event_timestamp < '2024-12-01'
        group by campaign_id
        order by CTR DESC
        limit 5) as top_c
        left join campaigns c on top_c.campaign_id = c.campaign_id
        order by top_c.CTR DESC;
    """,

    "2. ADVERTISER SPENDING": """
        WITH Campaign_Agg AS (
            SELECT campaign_id, SUM(ad_cost) AS camp_cost,
            SUM(CASE WHEN ad_revenue > 0 THEN 1 ELSE 0 END) AS camp_clicks,
            COUNT(event_id) AS camp_events
            FROM events WHERE event_timestamp >= '2024-11-01' AND event_timestamp < '2024-12-01'
            GROUP BY campaign_id
        ),
        Top_Advertisers AS (
            SELECT c.advertiser_id, SUM(ca.camp_cost) AS total_cost,
            SUM(ca.camp_clicks) AS total_clicks, SUM(ca.camp_events) AS total_events
            FROM Campaign_Agg ca JOIN campaigns c ON ca.campaign_id = c.campaign_id
            GROUP BY c.advertiser_id ORDER BY total_cost DESC LIMIT 5
        )
        SELECT a.advertiser_name, ta.total_cost,
        ROUND((ta.total_clicks / NULLIF(ta.total_events, 0)) * 100, 2) AS CTR
        FROM Top_Advertisers ta JOIN advertisers a ON ta.advertiser_id = a.advertiser_id
        ORDER BY ta.total_cost DESC;
    """,

    "3. COST EFFICIENCY (CPM & CPC)": """
        WITH Campaign_Agg AS (
            SELECT location_id, campaign_id, SUM(ad_cost) AS camp_cost, SUM(ad_revenue) AS camp_revenue,
            SUM(CASE WHEN ad_revenue > 0 THEN 1 ELSE 0 END) AS camp_clicks, COUNT(event_id) AS camp_events
            FROM events WHERE event_timestamp >= '2024-11-01' AND event_timestamp < '2024-12-01'
            GROUP BY location_id, campaign_id
        ),
        Top_Countries AS (
            SELECT l.country_name, ca.campaign_id, SUM(ca.camp_cost) AS total_cost,
            SUM(ca.camp_revenue) AS total_revenue, SUM(ca.camp_clicks) AS total_clicks, SUM(ca.camp_events) AS total_events
            FROM Campaign_Agg ca JOIN locations l ON l.location_id = ca.location_id
            GROUP BY l.country_name, ca.campaign_id ORDER BY total_revenue DESC LIMIT 30
        )
        SELECT tc.country_name, tc.campaign_id, tc.total_revenue, tc.total_cost,
        ROUND((tc.total_cost / NULLIF(tc.total_events, 0)) * 1000, 2) AS CPM,
        ROUND((tc.total_cost / NULLIF(tc.total_clicks, 0)), 2) AS CPC
        FROM Top_Countries tc ORDER BY tc.total_revenue DESC;
    """,

    "4. REGIONAL ANALYSIS (ROAS)": """
        WITH Location_Revenue AS (
            SELECT location_id, SUM(ad_revenue) AS total_revenue, SUM(ad_cost) AS total_cost
            FROM events WHERE event_timestamp >= '2024-11-01' AND event_timestamp < '2024-12-01'
            GROUP BY location_id HAVING SUM(ad_revenue) > 0 ORDER BY total_revenue DESC LIMIT 10
        )
        SELECT l.country_name, lr.total_revenue,
        ROUND((lr.total_revenue / NULLIF(lr.total_cost, 0)), 2) AS ROAS
        FROM Location_Revenue lr JOIN locations l ON lr.location_id = l.location_id
        ORDER BY lr.total_revenue DESC;
    """,

    "5. MOST ENGAGED USERS": """
        SELECT user_id, COUNT(event_id) AS total_impressions,
        SUM(CASE WHEN ad_revenue > 0 THEN 1 ELSE 0 END) AS total_clicks,
        ROUND((SUM(CASE WHEN ad_revenue > 0 THEN 1 ELSE 0 END) / NULLIF(COUNT(event_id), 0)) * 100, 2) AS user_CTR
        FROM events WHERE event_timestamp >= '2024-11-01' AND event_timestamp < '2024-12-01'
        GROUP BY user_id HAVING total_clicks > 0 ORDER BY total_clicks DESC, user_CTR DESC LIMIT 20;
    """,

    "6. BUDGET CONSUMPTION (>80%)": """
        WITH Campaign_Spend AS (
            SELECT campaign_id, SUM(ad_cost) AS total_spent
            FROM events WHERE event_timestamp >= '2024-10-01' AND event_timestamp < '2025-12-01'
            GROUP BY campaign_id
        )
        SELECT c.campaign_name, c.budget, cs.total_spent, c.remaining_budget,
        ROUND((cs.total_spent / NULLIF(c.budget, 0)) * 100, 2) AS budget_used_pct
        FROM Campaign_Spend cs JOIN campaigns c ON cs.campaign_id = c.campaign_id
        WHERE (cs.total_spent / NULLIF(c.budget, 0)) > 0.8 ORDER BY budget_used_pct DESC;
    """,

    "7. DEVICE PERFORMANCE": """
        SELECT e.device, COUNT(e.event_id) AS total_impressions,
        SUM(CASE WHEN e.ad_revenue > 0 THEN 1 ELSE 0 END) AS total_clicks,
        ROUND((SUM(CASE WHEN e.ad_revenue > 0 THEN 1 ELSE 0 END) / NULLIF(COUNT(e.event_id), 0)) * 100, 2) AS CTR
        FROM events e WHERE e.event_timestamp >= '2024-10-01' AND e.event_timestamp < '2024-12-01'
        GROUP BY device ORDER BY CTR DESC;
    """
}


In [7]:

# Процедура виконання та виводу результатів
try:
    with engine.connect() as conn:
        for title, sql in queries.items():
            # Виводимо заголовок звіту
            print("\n" + "="*60)
            print(f" ПОШУК ДАНИХ: {title}")
            print("="*60)

            # Читаємо SQL прямо в Pandas DataFrame
            df = pd.read_sql(sql, conn)

            if df.empty:
                print(f"Запит для '{title}' повернув порожній результат.")
            else:
                # Виводимо перші 10 рядків (head) для кожного звіту
                print(df.head(10).to_string(index=False))
                print(f"\n[Всього знайдено рядків: {len(df)}]")

            # текст у консоль
            sys.stdout.flush()

except Exception as e:
    print(f"\nПомилка під час виконання: {e}")

finally:
    # закриваємо пул з'єднань
    engine.dispose()
    print("\n" + "-"*60)
    print("Всі запити оброблені, з'єднання з базою закрито.")


 ПОШУК ДАНИХ: 1. CAMPAIGN PERFORMANCE (Top 5 by CTR)
campaign_name   CTR  total_cost
 Campaign_677 16.13       60.46
 Campaign_100  9.84      280.32
 Campaign_730  9.00      233.87
 Campaign_177  8.89       99.24
 Campaign_797  8.70      145.84

[Всього знайдено рядків: 5]

 ПОШУК ДАНИХ: 2. ADVERTISER SPENDING
advertiser_name  total_cost  CTR
  Advertiser_60    17656.82 4.97
  Advertiser_91    17649.72 4.82
  Advertiser_37    17268.17 4.94
  Advertiser_45    16925.92 5.00
  Advertiser_88    16915.32 5.45

[Всього знайдено рядків: 5]

 ПОШУК ДАНИХ: 3. COST EFFICIENCY (CPM & CPC)
country_name  campaign_id  total_revenue  total_cost     CPM   CPC
     Germany          510         100.28      466.79 2345.68 31.12
     Germany          144          98.61      377.98 2276.99 25.20
     Germany          241          90.84      265.98 2180.16 16.62
         USA          539          89.27      376.24 2381.27 28.94
     Germany          578          85.91      423.11 2299.51 28.21
   Australia

In [9]:
# Словник, куди будемо збирати всі дані для JSON
all_results_dict = {}

try:
    with engine.connect() as conn:
        for title, sql in queries.items():
            print(f"Обробка: {title}...")

            df = pd.read_sql(sql, conn)

            if not df.empty:
                # Перетворюємо таблицю у список словників (формат 'records')
                # orient='records' робить чистий JSON: [{"col1": val1, "col2": val2}, ...]
                all_results_dict[title] = df.to_dict(orient='records')
            else:
                all_results_dict[title] = []

    # Зберігаємо все у файл
    with open('analytics_results.json', 'w', encoding='utf-8') as f:
        json.dump(all_results_dict, f, ensure_ascii=False, indent=4)

    print("\n" + "="*60)
    print("Успіх! Всі таблиці збережені у файл: analytics_results.json")
    print("="*60)

except Exception as e:
    print(f"\nПомилка: {e}")

finally:
    engine.dispose()

Обробка: 1. CAMPAIGN PERFORMANCE (Top 5 by CTR)...
Обробка: 2. ADVERTISER SPENDING...
Обробка: 3. COST EFFICIENCY (CPM & CPC)...
Обробка: 4. REGIONAL ANALYSIS (ROAS)...
Обробка: 5. MOST ENGAGED USERS...
Обробка: 6. BUDGET CONSUMPTION (>80%)...
Обробка: 7. DEVICE PERFORMANCE...

Успіх! Всі таблиці збережені у файл: analytics_results.json
